# Strata on Kaggle — tuned, cache-first (C++ engine)

Same model, same server, with every lever we measured applied, and with **the engine, the pack and the MTP layer reused** from your own private Kaggle datasets instead of being rebuilt.

Kaggle wipes `/root` between sessions, so every step below asks the same question first: *is it already here?* Only when it is not does it restore from a dataset, and only when the dataset cannot supply it does it touch the network.

| Artifact | 1. session | 2. dataset | 3. network |
|---|---|---|---|
| Strata checkout | `setup.sh` exists | — | `git clone --depth 1` |
| `.venv` | `bin/python` exists | — | create |
| engine | `BUILD.json` + `strata` usable | `strata-runtime-cache` | ready-made download |
| cloudflared | `/usr/local/bin/cloudflared` | `strata-runtime-cache` | GitHub release |
| model GGUF | symlinks present | — | error: mount missing |
| pack | `experts.bin` + `index.txt` | `strata-<tag>-pack` | build (20–40 min) |
| MTP layer | `mtp/rt/experts.bin` | `strata-mtp-rt` | fetch 5.2 GB |
| tuning preset | `~/.config/strata/presets` | `strata-runtime-cache` (`tuning/`) | `strata tune auto` (once) |

## Engine acquisition — `ENGINE_SOURCE`

| Value | Behaviour |
|---|---|
| `"auto"` (default) | use the session copy → restore from `strata-runtime-cache` **when its version matches the repo's or is newer** → otherwise let setup download the matching ready-made engine |
| `"prebuilt"` | skip the cache and download the ready-made engine (moves any installed engine aside; use after an engine update) |
| `"build"` | compile the engine here from this checkout (`setup.sh --build`): 10–20 min, installs a CUDA toolkit if missing |

A cached engine **older than the checkout** is never restored — that is the "version match, otherwise download" rule. (A newer one is kept because setup itself accepts any engine at or above its floor, so rejecting it would only re-download every session.) A restored engine is also checksummed and checked for `sm_75` code before it is used.

## Engine tuning — `RUN_TUNE`

`strata tune auto` measures this machine's settings and saves them as a preset; the server applies the active preset at every start. Nothing here trains weights.

| Value | Behaviour |
|---|---|
| `"auto"` (default) | use a cached preset; with none, or after an engine/model change, run one full tune (a changed engine is rechecked in ~2 min first) |
| `"recheck"` | only recheck a cached preset; never a full tune |
| `"off"` | never tune |

The preset is stored in `strata-runtime-cache` under `tuning/presets/`, and Strata itself refuses to apply one whose engine, GPU, memory or model file changed. A tune is capped by `TUNE_MINUTES`; on failure the standard settings stay.

## What is different from `strata-in-kaggle.ipynb`

| Change | Why |
|---|---|
| Engine restored from a dataset | Skips the 1–3 min download, and cannot start an engine for the wrong arch |
| `--context 65536` (was 262144) | q4_0 KV is ~1.96 GB at 256K vs ~0.49 GB at 64K → ~1.5 GB more expert cache |
| `--vram-reserve-mib 0` | Kaggle has no desktop; the default 700 MiB/card is wasted |
| `--draft-vocab en` | English+code draft subset: ~110 MiB VRAM, 1–2% decode |
| `--vision no` by default | No encoder mapping or CPU work unless you send images |
| `--mtp on` | An accepted draft skips a whole set of expert reads — worth more on a disk-bound box |
| Pack + MTP restored from datasets | Skips the ~50 GB `experts.bin` build and the 5.2 GB draft fetch |
| `strata tune` preset cached | Measures the engine once; later sessions start with the winning flags at no cost |
| Config pinned to `strata-<tag>.json` | The old `sorted(glob(...))[0]` could start the wrong model |
| Tunnel no longer blocks the kernel | `cloudflared` runs in the background; the cell returns |
| `STRATA_REQUEST_LINES=1` | One log line per request with prefill/decode numbers |
| No Drive backup, no `input()` prompts | Works with *Run All*; no stale-config restore |

The Rust engine (RustingStrata) is deliberately not used: its CUDA kernels need `cp.async` and `m16n8k32`, which ptxas rejects for `sm_75`. It cannot start on a T4.

## Prerequisites

1. Session options: **GPU T4 ×2** and **Internet on**.
2. Secrets (Add-ons → Secrets), all optional:
   - `API_KEY` — recommended. It turns off the tunnel Host check and protects the endpoint.
   - `KAGGLE_API_TOKEN` — only needed to build caches (upload) with the `kaggle` CLI.
   - `CLOUDFLARE_TUNNEL_TOKEN` — only for a named tunnel instead of a quick tunnel. A token run ignores a local `config.yml`, so the hostname must be a **Public Hostname** on that tunnel (service `HTTP` at `127.0.0.1:8080`) and `PUBLIC_HOSTNAME` must match it.
3. Inputs (Add-ons → Add Input), attach what you have:
   - the model mount, e.g. `buianhtruc/strata-qwen3-8-flash-next-gsq-rco-iq3-s`
   - `strata-runtime-cache` — engine + cloudflared (skips the download)
   - `strata-<tag>-pack` — skips the 50 GB build
   - `strata-mtp-rt` — skips the 5.2 GB draft fetch

## Expected timeline

| Step | First session (no caches) | Later sessions (cached) |
|---|---|---|
| clone + venv | 2–4 min | seconds |
| engine | 1–3 min (download) | ~20 s (restore) |
| engine build (`ENGINE_SOURCE="build"`) | 10–20 min + toolkit | — |
| model symlinks | seconds | seconds |
| pack + MTP restore | — | ~1 min (symlinks) |
| **`experts.bin` build** | **20–40 min** | **skipped** |
| **MTP fetch + pack** | **5–10 min** | **skipped** |
| `strata tune` (`RUN_TUNE="auto"`, first time) | +20–40 min | — (preset cached) |
| setup total | ~30–55 min | ~3–5 min |

Run the cells top to bottom once, then set `UPLOAD_CACHE = True` in the config cell and re-run only the last cell to publish the caches.


## 1. Configuration — edit only this cell

Edit only this cell. It sets the model, the tuned launch flags, the engine acquisition mode (`ENGINE_SOURCE`), the cache dataset names and the secrets. Everything below reads these values.

In [ ]:
# ==================== EDIT ONLY THIS CELL ====================
KAGGLE_USERNAME = "kvinodhkumar"  # your Kaggle username (owner of the cache datasets)

FAMILY = "qwen"     # "qwen" (general IQ quants) | "coder" (code) | "gyro"
MODEL  = "IQ3_S"    # qwen: IQ3_S / IQ3_XXS / Q2_0 | coder: IQ1_M | gyro: Gyro-M / Gyro-S

# ---- tuned launch settings ----
CONTEXT          = 65536   # 262144 costs ~1.5 GB of expert cache for headroom you do not need
KV               = "q4_0"  # 4-bit KV: ~7.5 KB per context token
VISION           = "no"    # "yes" for images (CPU encoder, slower, a little RAM)
GPUS             = "0,1"
VRAM_RESERVE_MIB = 0       # no desktop on Kaggle: hand the 700 MiB/card to the expert cache
DRAFT_VOCAB      = "en"    # English + code draft subset (qwen/coder only)
MTP              = "on"    # keep speculative decoding
PORT             = 8080

# ---- engine acquisition ----
# "auto"     session -> dataset (version match or newer) -> ready-made download
# "prebuilt" skip the cache and download the ready-made engine
# "build"    compile it here from this checkout (10-20 min, installs a CUDA toolkit if missing)
ENGINE_SOURCE  = "auto"
ENGINE_VERSION = "0.1.57"  # pinned; "" = read project(strata VERSION ...) from the checkout

# ---- engine tuning (strata tune) ----
# "off"     never tune: use the flags above as they are
# "recheck" use a cached preset; when the engine changed, run the ~2 min recheck (never a full tune)
# "auto"    use a cached preset; when there is none (or the engine/model changed) run one full tune
RUN_TUNE     = "auto"
TUNE_MINUTES = 25          # cap for a full tune (quick mode); 0 = the tuner's own estimate
TUNE_USE     = "coding"    # "chat" or "coding": what it judges long-prompt speed on

# ---- derived names (do not edit) ----
_TAG        = {"qwen": "", "coder": "coder-", "gyro": "gyro-"}[FAMILY] + MODEL
SLUG        = _TAG.lower()                       # iq3_s / coder-iq1_m / gyro-gyro-m (local paths)
DATASET_TAG = SLUG.replace("_", "-")             # Kaggle slugs: letters, digits and dashes only
STRATA_DIR  = "/root/Strata"
DATA_DIR    = "/root/Strata-data"
CONFIG_PATH = f"{STRATA_DIR}/strata-{SLUG}.json"
LOG_PATH    = "/kaggle/working/strata_server.log"
CF_LOG_PATH = "/kaggle/working/cloudflared.log"
MODEL_DIR   = f"{DATA_DIR}/models/{_TAG}"
PACK_DIR    = f"{DATA_DIR}/packs/{SLUG}"

# ---- model mount (read-only Kaggle input) ----
MODEL_MOUNTS = {
    ("qwen",  "IQ3_S"):  "/kaggle/input/models/buianhtruc/strata-qwen3-8-flash-next-gsq-rco-iq3-s/gguf/default/1",
    ("gyro",  "Gyro-M"): "/kaggle/input/models/buianhtruc/strata-qwen3-8-flash-next-gyro-m-tq2-0/gguf/default/1",
    ("gyro",  "Gyro-S"): None,
    ("coder", "IQ1_M"):  None,
}
MODEL_MOUNT = MODEL_MOUNTS.get((FAMILY, MODEL))

# ---- private cache datasets (set any to None to always rebuild that part) ----
RUNTIME_DATASET = f"{KAGGLE_USERNAME}/strata-runtime-cache"
PACK_DATASET    = f"{KAGGLE_USERNAME}/strata-{DATASET_TAG}-pack"
MTP_DATASET     = f"{KAGGLE_USERNAME}/strata-mtp-rt"

# ---- one-time: flip to True in the session that builds a cache ----
UPLOAD_CACHE = False

# ---- secrets ----
API_KEY_SECRET      = "API_KEY"
KAGGLE_TOKEN_SECRET = "KAGGLE_API_TOKEN"
TUNNEL_TOKEN_SECRET = "CLOUDFLARE_TUNNEL_TOKEN"
PUBLIC_HOSTNAME     = "https://strata.cyberctl.com"   # named tunnel host (scheme optional)

ENABLE_TUNNEL    = True
HEALTH_TIMEOUT_S = 1800      # 30 min: filling the expert cache from the input mount is slow


## 2. Preflight — what will run, and is the mount attached?

Checks the mount, lists the GGUF shards, and prints the planned action for every artifact (reuse / restore / rebuild). If the mount is missing, attach it (Add-ons -> Add Input) before continuing.

In [ ]:
from pathlib import Path
import shutil

print("model  :", FAMILY, MODEL)
print("tag    :", SLUG)
print("config :", CONFIG_PATH)
print("pack   :", PACK_DIR)
print("engine :", ENGINE_SOURCE)
print("mount  :", MODEL_MOUNT or "(none configured - set MODEL_MOUNTS or MODEL_MOUNT)")
print(f"free   : {shutil.disk_usage('/root').free / 1e9:.1f} GB on /root")

owned = sorted(p.name for p in Path("/kaggle/input/datasets").glob("*")) \
    if Path("/kaggle/input/datasets").is_dir() else []
if owned and KAGGLE_USERNAME not in owned:
    print("!! the attached datasets belong to", ", ".join(owned), "- not", KAGGLE_USERNAME)
    print("   set KAGGLE_USERNAME to your own account: the pack, MTP and runtime names are")
    print("   derived from it, and an upload under another owner is refused (403)")

if MODEL_MOUNT is None:
    print("!! no mount configured for this model: the 58 GB download would land in the writable layer")
elif not Path(MODEL_MOUNT).exists():
    print("!! the model mount is NOT attached - Add-ons > Add Input")
else:
    ggufs = sorted(Path(MODEL_MOUNT).glob("*.gguf"))
    total = sum(p.stat().st_size for p in ggufs) / 1e9
    print(f"mount OK: {len(ggufs)} gguf file(s), {total:.1f} GB")
    for p in ggufs:
        print("   ", p.name, f"{p.stat().st_size/1e9:.1f} GB")
    shards = [p for p in ggufs if "-00001-of-" in p.name]
    if shards:
        print("   first shard:", shards[0].name, "(pass this one to the engine)")


## 3. Inventory — what is already here, and where the rest will come from

Probes the session and any attached datasets and prints `artifact | state | action`. Advisory only - the engine cell below is authoritative.

In [ ]:
import json
import re
import tarfile
from pathlib import Path

STRATA = Path(STRATA_DIR)
DATA = Path(DATA_DIR)

def dataset_dir(dataset, marker):
    """Find an attached dataset by owner/slug, probing the layouts Kaggle uses."""
    if not dataset:
        return None
    owner, _, slug = dataset.rpartition("/")
    for c in (Path("/kaggle/input") / dataset,
              Path("/kaggle/input") / slug,
              Path("/kaggle/input/datasets") / dataset,
              Path("/kaggle/input") / owner / slug):
        if (c / marker).exists():
            return c
    return None

def repo_engine_version():
    try:
        text = (STRATA / "CMakeLists.txt").read_text(encoding="utf-8")
    except OSError:
        return ""
    m = re.search(r"project\(strata VERSION ([\d.]+)", text)
    return m.group(1) if m else ""

def engine_meta(folder):
    try:
        return json.loads((folder / "BUILD.json").read_text(encoding="utf-8"))
    except Exception:
        return {}

engine_archs_KEYS = ("archs", "engine_archs", "cuda_archs", "cmake_cuda_architectures",
              "CUDA_ARCHITECTURES", "TORCH_CUDA_ARCH_LIST", "torch_cuda_arch_list")

def engine_archs(meta):
    value = None
    for key in engine_archs_KEYS:
        if meta.get(key) not in (None, ""):
            value = meta.get(key)
            break
    if value in (None, ""):
        return set()
    parts = list(value) if isinstance(value, (list, tuple, set)) else re.split(r"[;,\s]+", str(value))
    out = set()
    for part in parts:
        text = str(part).strip().lower()
        # 75, sm_75, 7.5 and 8.6a all name a compute capability
        match = re.fullmatch(r"(?:sm_)?(\d+)(?:\.(\d+))?[a-z]*", text)
        if match:
            out.add(int(match.group(1)) * 10 + int(match.group(2)) if match.group(2) else int(match.group(1)))
            continue
        fallback = re.search(r"\d+", text)
        if fallback:
            out.add(int(fallback.group()))
    return out

def ver_tuple(value):
    """0.1.57 -> (0, 1, 57); "" or junk -> () (unknown, treated as acceptable)."""
    parts = re.findall(r"\d+", str(value or ""))
    return tuple(int(p) for p in parts[:4])

EXPECTED = ENGINE_VERSION or repo_engine_version()
EXPECTED_T = ver_tuple(EXPECTED)
runtime_ds = dataset_dir(RUNTIME_DATASET, "runtime-manifest.json")
cached_meta = {}
if runtime_ds is not None:
    try:
        cached_meta = json.loads((runtime_ds / "runtime-manifest.json").read_text())
    except Exception:
        cached_meta = {}
cached_version = str(cached_meta.get("engine_version") or cached_meta.get("strata_version") or "")
cached_archs = engine_archs(cached_meta)

def archive_members(path, limit=200):
    """Leading member names, stopping as soon as engine/BUILD.json is seen."""
    try:
        with tarfile.open(path) as tf:
            names = []
            for member in tf:
                names.append(member.name)
                if member.name == "engine/BUILD.json" or len(names) >= limit:
                    break
            return names
    except Exception:
        return []

runtime_archive = None
if runtime_ds is not None:
    for name in ("strata-runtime-cache.bin", "strata-runtime-cache.tar.gz"):
        if (runtime_ds / name).is_file():
            runtime_archive = runtime_ds / name
            break
# This notebook restores engine/ + third_party/llama.cpp + tools/cloudflared.
# A cache written elsewhere holds terminal/ + components/ and cannot be used.
layout_ok = bool(runtime_archive is not None and "engine/BUILD.json" in archive_members(runtime_archive))

cache_ok = bool(runtime_ds is not None and layout_ok
                and (not EXPECTED_T or not cached_version or ver_tuple(cached_version) >= EXPECTED_T)
                and (not cached_archs or 75 in cached_archs)
                and cached_meta.get("engine_source") != "local")

engine_present = (STRATA / "engine" / "strata").is_file() and (STRATA / "engine" / "BUILD.json").is_file()
engine_ok = False
if engine_present:
    meta = engine_meta(STRATA / "engine")
    ver = str(meta.get("version", ""))
    engine_ok = (meta.get("source") != "local"
                 and (not EXPECTED_T or ver_tuple(ver) >= EXPECTED_T)
                 and 75 in engine_archs(meta))

local_build = engine_present and str(engine_meta(STRATA / "engine").get("source")) == "local"
if engine_ok:
    engine_action = "use it"
elif local_build and ENGINE_SOURCE != "build":
    engine_action = "local build - setup reuses or rebuilds"
elif ENGINE_SOURCE == "build":
    engine_action = "compile here (10-20 min)"
elif ENGINE_SOURCE == "prebuilt":
    engine_action = "download (prebuilt)"
elif cache_ok:
    engine_action = "restore " + RUNTIME_DATASET + ("" if EXPECTED else " (version re-checked after the clone)")
elif runtime_ds is not None and cached_meta.get("engine_source") == "local":
    engine_action = "cache holds a local build -> download"
elif runtime_ds is not None and cached_archs and 75 not in cached_archs:
    engine_action = f"cache has archs {sorted(cached_archs)}, no sm_75 -> download"
elif runtime_ds is not None and EXPECTED_T and cached_version and ver_tuple(cached_version) < EXPECTED_T:
    engine_action = f"cache {cached_version} is older than {EXPECTED} -> download"
elif runtime_ds is not None and runtime_archive is None:
    engine_action = "cache has no archive -> download"
elif runtime_ds is not None and not layout_ok:
    engine_action = "cache has terminal/components, no engine/ -> download"
else:
    engine_action = "download (prebuilt)"

rows = [
    ("checkout",    "present" if (STRATA / "setup.sh").exists() else "absent",
                    "use it" if (STRATA / "setup.sh").exists() else "git clone"),
    ("venv",        "present" if (STRATA / ".venv" / "bin" / "python").exists() else "absent",
                    "use it" if (STRATA / ".venv" / "bin" / "python").exists() else "create"),
    ("engine",      ("present v" + str(engine_meta(STRATA / "engine").get("version", "?"))) if engine_present else "absent",
                    engine_action),
    ("cloudflared", "present" if Path("/usr/local/bin/cloudflared").is_file() else "absent",
                    "use it" if Path("/usr/local/bin/cloudflared").is_file() else
                    ("restore " + RUNTIME_DATASET if cache_ok else "download")),
    ("model",       (str(len(list(Path(MODEL_MOUNT).glob("*.gguf"))) ) + " gguf") if MODEL_MOUNT and Path(MODEL_MOUNT).exists() else "absent",
                    "link from the mount" if MODEL_MOUNT else "set MODEL_MOUNTS"),
    ("pack",        "present" if (Path(PACK_DIR) / "experts.bin").exists() and (Path(PACK_DIR) / "index.txt").exists() else "absent",
                    "use it" if (Path(PACK_DIR) / "experts.bin").exists() and (Path(PACK_DIR) / "index.txt").exists()
                    else ("restore " + PACK_DATASET if dataset_dir(PACK_DATASET, "manifest.json") else "build (20-40 min)")),
    ("mtp",         "present" if Path(DATA_DIR, "mtp", "rt", "experts.bin").exists() else "absent",
                    "use it" if Path(DATA_DIR, "mtp", "rt", "experts.bin").exists()
                    else ("restore " + MTP_DATASET if dataset_dir(MTP_DATASET, "manifest.json") else "fetch 5.2 GB")),
]

print(f"engine version wanted: {EXPECTED or '(unknown until the checkout exists)'}")
print("Advisory only: each action runs in the cell named in the last column.")
print("Nothing is linked or downloaded until you run that cell.")
print()
print(f"{'artifact':12} {'state':16} {'action':56} step")
print("-" * 96)
STEPS = {"checkout": "cell 10", "venv": "cell 10", "engine": "cell 12", "cloudflared": "cell 12",
         "model": "cell 14", "pack": "cell 16", "mtp": "cell 16"}
for name, state, action in rows:
    print(f"{name:12} {state:16} {action:56} {STEPS.get(name, '')}")


## 4. Secrets

Reads `API_KEY`, `KAGGLE_API_TOKEN`, `CLOUDFLARE_TUNNEL_TOKEN` from Kaggle Secrets (Add-ons -> Secrets). All optional, but `API_KEY` is strongly recommended.

In [ ]:
from kaggle_secrets import UserSecretsClient

_sec = UserSecretsClient()

def _secret(name):
    try:
        value = _sec.get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None

API_KEY      = _secret(API_KEY_SECRET)
KAGGLE_TOKEN = _secret(KAGGLE_TOKEN_SECRET)
TUNNEL_TOKEN = _secret(TUNNEL_TOKEN_SECRET)

print("API_KEY      :", "set - host check off, endpoint authenticated" if API_KEY
      else "NOT set - the tunnel will be open to anyone with the URL")
print("KAGGLE token :", "set" if KAGGLE_TOKEN else "not set (only needed to upload caches)")
print("Tunnel token :", "set - named tunnel" if TUNNEL_TOKEN else "not set - quick tunnel")


## 5. Strata checkout and venv — reused when present

Clones Strata (shallow) and creates `.venv` only when missing. Both are reused on re-run; `.venv` is never cached between sessions.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

STRATA = Path(STRATA_DIR)

if (STRATA / "setup.sh").exists():
    print("Strata checkout: reused")
else:
    if STRATA.exists():
        shutil.rmtree(STRATA)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Niko1221/Strata.git", str(STRATA)], check=True)
    print("Strata checkout: cloned")

VENV_PY = STRATA / ".venv" / "bin" / "python"
if VENV_PY.exists():
    print("venv: reused")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(STRATA / ".venv")], check=True)
    subprocess.run([str(VENV_PY), "-m", "pip", "install", "-q", "wrapt"], check=True)
    print("venv: created")


## 6. Engine and cloudflared — check, then restore or download

Acquires the engine and `cloudflared` for this session. Order: session copy -> `strata-runtime-cache` dataset (only if its version is not older than the checkout, it is not a local build, it has `sm_75` code and its `.sha256` matches) -> ready-made download. `ENGINE_SOURCE` overrides this. The archive is an uncompressed tar named `.bin` (Kaggle auto-unpacks `.tar.gz`); it contains `engine/`, `third_party/llama.cpp`, `tools/cloudflared`. Exec bits are restored with `chmod 755`.

`ENGINE_SOURCE="build"` makes setup compile the engine (10-20 min, installs a CUDA toolkit if missing).


In [ ]:
import hashlib
import json
import os
import re
import shutil
import tarfile
import urllib.request
from pathlib import Path

STRATA = Path(STRATA_DIR)
ENGINE = STRATA / "engine"
CF = Path("/usr/local/bin/cloudflared")
CF_URL = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
          "cloudflared-linux-amd64")
ARCHIVE_NAMES = ("strata-runtime-cache.bin", "strata-runtime-cache.tar.gz")

def runtime_archive(ds):
    """The runtime tar in the dataset.  `.bin` is an uncompressed tar: Kaggle
    unpacks archive extensions server-side, which would scatter the tree."""
    if ds is None:
        return None
    for name in ARCHIVE_NAMES:
        if (ds / name).is_file():
            return ds / name
    return None

def dataset_dir(dataset, marker):
    if not dataset:
        return None
    owner, _, slug = dataset.rpartition("/")
    for c in (Path("/kaggle/input") / dataset,
              Path("/kaggle/input") / slug,
              Path("/kaggle/input/datasets") / dataset,
              Path("/kaggle/input") / owner / slug):
        if (c / marker).exists():
            return c
    return None

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def repo_version():
    try:
        text = (STRATA / "CMakeLists.txt").read_text(encoding="utf-8")
    except OSError:
        return ""
    m = re.search(r"project\(strata VERSION ([\d.]+)", text)
    return m.group(1) if m else ""

archs_of_KEYS = ("archs", "engine_archs", "cuda_archs", "cmake_cuda_architectures",
              "CUDA_ARCHITECTURES", "TORCH_CUDA_ARCH_LIST", "torch_cuda_arch_list")

def archs_of(meta):
    value = None
    for key in archs_of_KEYS:
        if meta.get(key) not in (None, ""):
            value = meta.get(key)
            break
    if value in (None, ""):
        return set()
    parts = list(value) if isinstance(value, (list, tuple, set)) else re.split(r"[;,\s]+", str(value))
    out = set()
    for part in parts:
        text = str(part).strip().lower()
        # 75, sm_75, 7.5 and 8.6a all name a compute capability
        match = re.fullmatch(r"(?:sm_)?(\d+)(?:\.(\d+))?[a-z]*", text)
        if match:
            out.add(int(match.group(1)) * 10 + int(match.group(2)) if match.group(2) else int(match.group(1)))
            continue
        fallback = re.search(r"\d+", text)
        if fallback:
            out.add(int(fallback.group()))
    return out

def ver_tuple(value):
    """0.1.57 -> (0, 1, 57); "" or junk -> () (unknown, treated as acceptable)."""
    parts = re.findall(r"\d+", str(value or ""))
    return tuple(int(p) for p in parts[:4])

EXPECTED = ENGINE_VERSION or repo_version()
EXPECTED_T = ver_tuple(EXPECTED)

def engine_info():
    try:
        return json.loads((ENGINE / "BUILD.json").read_text(encoding="utf-8"))
    except Exception:
        return {}

def engine_state():
    exe = ENGINE / "strata"
    if not (exe.is_file() and (ENGINE / "BUILD.json").is_file()):
        return False, "not installed"
    info = engine_info()
    if info.get("source") == "local":
        return False, "local build"
    have = str(info.get("version", "?"))
    if EXPECTED_T and ver_tuple(have) < EXPECTED_T:
        return False, f"version {have} is older than the checkout's {EXPECTED}"
    known = archs_of(info)
    if known and 75 not in known:
        return False, f"no sm_75 code (has {sorted(known)})"
    note = "" if not EXPECTED_T or ver_tuple(have) == EXPECTED_T else f" (newer than the checkout's {EXPECTED})"
    return True, f"version {have}, archs {sorted(known) if known else 'unknown'}{note}"

print("engine version wanted:", EXPECTED or "(unknown)")

if ENGINE_SOURCE == "build":
    print("engine: ENGINE_SOURCE=build -> setup compiles it (10-20 min, CUDA toolkit required)")
elif ENGINE_SOURCE == "prebuilt":
    print("engine: ENGINE_SOURCE=prebuilt -> setup downloads the ready-made engine")
    if ENGINE.exists():
        backup = ENGINE.with_name("engine.before-prebuilt")
        shutil.rmtree(backup, ignore_errors=True)
        shutil.move(str(ENGINE), str(backup))
        print("  moved the installed engine to", backup.name, "so setup downloads a fresh one")
else:
    ok, why = engine_state()
    local = (ENGINE / "strata").is_file() and engine_info().get("source") == "local"
    ds = dataset_dir(RUNTIME_DATASET, "runtime-manifest.json")
    archive = runtime_archive(ds)
    if archive is None:
        ds = None
    print("engine:", ("usable -> " + why) if ok else why)
    if ok:
        pass
    elif local:
        print("  local build present -> left in place; setup reuses it or rebuilds it")
    elif ds is None:
        print("  no", RUNTIME_DATASET, "with an archive attached -> setup downloads the matching engine")
    else:
        cached = {}
        try:
            cached = json.loads((ds / "runtime-manifest.json").read_text(encoding="utf-8"))
        except Exception:
            cached = {}
        cached_version = str(cached.get("engine_version") or cached.get("strata_version") or "")
        checksum_file = Path(str(archive) + ".sha256")
        usable = True
        if checksum_file.is_file():
            want = checksum_file.read_text(encoding="utf-8").split()[0]
            if sha256(archive) != want:
                print("  cache checksum mismatch -> not restored; setup downloads")
                usable = False
        if usable and EXPECTED_T and cached_version and ver_tuple(cached_version) < EXPECTED_T:
            print(f"  cache holds {cached_version}, older than the checkout's {EXPECTED} -> not restored; setup downloads")
            usable = False
        if usable and cached.get("engine_archs") and 75 not in archs_of(cached):
            print(f"  cache has archs {sorted(archs_of(cached))}, no sm_75 -> not restored; setup downloads")
            usable = False
        if usable and cached.get("engine_source") == "local":
            print("  cache holds a local build -> not restored; setup downloads (or set ENGINE_SOURCE=build)")
            usable = False
        if usable:
            tmp = Path("/root/.engine-restore")
            shutil.rmtree(tmp, ignore_errors=True)
            tmp.mkdir(parents=True)
            try:
                with tarfile.open(archive) as tf:
                    # the archive is the user's own, checksummed: keep exec bits and
                    # any relative symlinks the engine ships with
                    try:
                        tf.extractall(tmp, filter="fully_trusted")
                    except TypeError:               # Python < 3.12
                        tf.extractall(tmp)
                cand = tmp / "engine"
                have, have_archs, have_source = "", set(), ""
                try:
                    info = json.loads((cand / "BUILD.json").read_text(encoding="utf-8"))
                    have = str(info.get("version", ""))
                    have_archs = archs_of(info)
                    have_source = str(info.get("source", ""))
                except Exception:
                    pass
                if not (cand / "BUILD.json").is_file():
                    inside = sorted(p.name for p in cand.iterdir())[:6] if cand.is_dir() else []
                    print("  archive has no engine/BUILD.json"
                          + (" (engine/ holds: " + ", ".join(inside) + ")" if inside else "")
                          + " -> setup downloads; re-run cell 30 to publish the metadata")
                elif EXPECTED_T and have and ver_tuple(have) < EXPECTED_T:
                    print(f"  archive holds {have}, older than the checkout's {EXPECTED} -> setup downloads")
                elif have_archs and 75 not in have_archs:
                    print(f"  archive has archs {sorted(have_archs)}, no sm_75 -> setup downloads")
                elif have_source == "local":
                    print("  archive holds a local build -> setup downloads (or set ENGINE_SOURCE=build)")
                else:
                    if not have_archs:
                        print("  BUILD.json lists no archs; keeping the cached engine (it reports them at start)")
                    if ENGINE.exists():
                        shutil.rmtree(ENGINE)
                    shutil.move(str(cand), str(ENGINE))
                    for name in ("strata", "strata-vision"):
                        p = ENGINE / name
                        if p.is_file():
                            p.chmod(0o755)
                    llama = tmp / "third_party" / "llama.cpp"
                    if llama.is_dir() and not (STRATA / "third_party" / "llama.cpp").exists():
                        (STRATA / "third_party").mkdir(parents=True, exist_ok=True)
                        shutil.move(str(llama), str(STRATA / "third_party" / "llama.cpp"))
                    tool = tmp / "tools" / "cloudflared"
                    if tool.is_file():
                        shutil.copy2(tool, CF)
                        CF.chmod(0o755)
                    print("  restored ->", engine_state()[1])
            except Exception as exc:
                print("  restore failed:", exc, "-> setup downloads")
            finally:
                shutil.rmtree(tmp, ignore_errors=True)

PRESETS = Path(os.environ.get("XDG_CONFIG_HOME", str(Path.home() / ".config"))) / "strata" / "presets"
if not PRESETS.exists():
    tds = dataset_dir(RUNTIME_DATASET, "runtime-manifest.json")
    tarchive = runtime_archive(tds)
    restored = 0
    if tarchive is not None:
        tmp = Path("/root/.presets-restore")
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        try:
            with tarfile.open(tarchive) as tf:
                members = [m for m in tf.getmembers()
                           if m.name == "tuning/presets" or m.name.startswith("tuning/presets/")]
                if members:
                    try:
                        tf.extractall(tmp, members=members, filter="fully_trusted")
                    except TypeError:               # Python < 3.12
                        tf.extractall(tmp, members=members)
                    src = tmp / "tuning" / "presets"
                    if src.is_dir():
                        PRESETS.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copytree(src, PRESETS, dirs_exist_ok=True)
                        restored = len([p for p in PRESETS.glob("*.json") if p.name != "active.json"])
        except Exception as exc:
            print("tuning: preset restore skipped (", exc, ")")
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
    print("tuning: presets restored" if restored else "tuning: no cached presets (the tuning cell can measure some)")

if CF.is_file():
    print("cloudflared: present")
else:
    cf_ds = dataset_dir(RUNTIME_DATASET, "runtime-manifest.json")
    cf_archive = runtime_archive(cf_ds)
    restored = False
    if cf_archive is not None:
        try:
            with tarfile.open(cf_archive) as tf:
                member = tf.getmember("tools/cloudflared")
                tmp = Path("/tmp/cf-restore")
                shutil.rmtree(tmp, ignore_errors=True)
                tmp.mkdir(parents=True)
                try:
                    tf.extract(member, tmp, filter="fully_trusted")
                except TypeError:                  # Python < 3.12
                    tf.extract(member, tmp)
                shutil.copy2(tmp / "tools" / "cloudflared", CF)
                CF.chmod(0o755)
                restored = True
                print("cloudflared: restored from", cf_ds)
        except Exception:
            restored = False
    if not restored:
        try:
            CF.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(CF_URL, CF)
            CF.chmod(0o755)
            print("cloudflared: downloaded")
        except Exception as exc:
            print("cloudflared: download failed (", exc, ") - the tunnel cell will skip")


## 7. Model files — symlinked from the mount (never copied)

Creates symlinks in `DATA_DIR/models/<tag>` pointing at the read-only mount, so the ~84 GB model costs 0 bytes of the writable layer. Stale links are pruned.

In [ ]:
from pathlib import Path

if MODEL_MOUNT is None:
    print("skipped: no mount configured for", FAMILY, MODEL)
else:
    source = Path(MODEL_MOUNT)
    target = Path(MODEL_DIR)
    if not source.exists():
        raise FileNotFoundError(f"mount not attached: {source}")

    files = sorted(source.glob("*.gguf")) + sorted(source.glob("*.done"))
    if not files:
        raise FileNotFoundError(f"no GGUF files in {source}")

    target.mkdir(parents=True, exist_ok=True)
    wanted = {f.name for f in files}
    kept = changed = 0
    for f in files:
        link = target / f.name
        if link.is_symlink() and link.resolve() == f.resolve():
            kept += 1
            continue
        if link.is_symlink() or link.exists():
            link.unlink()
        link.symlink_to(f)
        changed += 1
    for link in target.iterdir():
        if link.is_symlink() and link.name not in wanted:
            link.unlink()
            changed += 1

    first = next((f for f in files if "-00001-of-" in f.name), files[0])
    print(f"{kept} link(s) already correct, {changed} updated")
    print("engine will read:", first.name)

    marks = [f for f in files if f.name.endswith(".gguf.done")]
    if not marks:
        print("no .done marks in the mount: setup will verify every shard's SHA-256 once")
        print("(84 GB read; adds several minutes, nothing is downloaded again)")


## 8. Restore the pack and the draft layer

Symlinks only — nothing is copied. When both are present, setup skips the ~50 GB `experts.bin` build and the 5.2 GB MTP fetch. If a dataset is not attached, the part is rebuilt by setup (slow, but correct).

In [ ]:
import json
from pathlib import Path

DATA = Path(DATA_DIR)

def dataset_dir(dataset):
    """Find the attached mount for owner/slug in the layouts Kaggle uses."""
    if not dataset:
        return None
    owner, _, slug = dataset.rpartition("/")
    candidates = (Path("/kaggle/input/datasets") / dataset,
                  Path("/kaggle/input") / dataset,
                  Path("/kaggle/input") / slug,
                  Path("/kaggle/input") / owner / slug)
    for c in candidates:
        if (c / "manifest.json").is_file():
            return c
    for c in candidates:
        if c.is_dir():
            return c
    return None

def restore(dataset):
    ds = dataset_dir(dataset)
    if ds is None:
        print(f"restore: {dataset} is NOT attached -> it will be rebuilt")
        print("  add it with Add-ons > Add Input, then re-run this cell")
        return 0
    manifest = ds / "manifest.json"
    if not manifest.is_file():
        print(f"restore: {ds} is attached but has no manifest.json at its top level")
        print("  it was built by a different tool, so its file names are unknown -> rebuilt")
        print("  contents:", ", ".join(sorted(p.name for p in list(ds.iterdir())[:10])))
        return 0
    entries = json.loads(manifest.read_text(encoding="utf-8"))["files"]
    print(f"restore: using {ds} ({len(entries)} entries)")
    linked = kept = 0
    for e in entries:
        src = ds / e["dataset_name"]
        dst = DATA / e["relative_path"]
        if not src.exists():
            print("  missing in dataset:", e["dataset_name"])
            continue
        if dst.is_symlink() and dst.resolve() == src.resolve():
            kept += 1
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        elif dst.is_dir():
            import shutil as _shutil
            _shutil.rmtree(dst)
        dst.symlink_to(src)
        linked += 1
    print(f"restore: {ds} -> {linked} linked, {kept} already correct ({len(entries)} entries)")
    return linked

pack = Path(PACK_DIR)
if (pack / "experts.bin").exists() and (pack / "index.txt").exists():
    print("pack: present")
else:
    restore(PACK_DATASET)

if Path(DATA_DIR, "mtp", "rt", "experts.bin").exists():
    print("MTP: present")
else:
    restore(MTP_DATASET)

pack = Path(PACK_DIR)
have_pack = (pack / "experts.bin").exists() and (pack / "index.txt").exists()
have_mtp = Path(DATA_DIR, "mtp", "rt", "experts.bin").exists()
print("\npack ready :", have_pack, "(else setup writes ~50 GB again)")
print("MTP ready  :", have_mtp, "(else setup fetches 5.2 GB again)")

# Attached is not linked: this shows, file by file, whether the symlink exists and
# whether its target is readable. A BROKEN line means the mount changed its names.
def describe(path):
    if path.is_symlink():
        target = path.resolve()
        return ("link -> " + str(target) + ("" if path.exists() else "   BROKEN"))
    if path.is_file():
        return "real file"
    return "directory" if path.is_dir() else "missing"

for label, base in (("pack", Path(PACK_DIR)), ("mtp", Path(DATA_DIR) / "mtp" / "rt")):
    print(f"\n{label}: {base}")
    if not base.is_dir():
        print("  the directory does not exist - the restore above did not link anything")
        continue
    for entry in sorted(base.iterdir())[:12]:
        print(f"  {entry.name:30} {describe(entry)}")


## 9. Install Strata with the tuned settings

Runs `setup.sh --setup` with the tuned flags. Restores the pack/MTP first so the ~50 GB `experts.bin` build is skipped when cached. Passes `--build` only when `ENGINE_SOURCE="build"`.

In [ ]:
import shlex
import subprocess
from pathlib import Path

_pack = Path(PACK_DIR)
if not ((_pack / "experts.bin").exists() and (_pack / "index.txt").exists()):
    print("!! the pack is not complete: setup will build ~50 GB of experts.bin (20-40 min)")
    print("   attach", PACK_DATASET, "and re-run the restore cell to skip that")

cmd = ["sh", f"{STRATA_DIR}/setup.sh", "--setup",
       "--family", FAMILY, "--model", MODEL,
       "--gpus", GPUS,
       "--context", str(CONTEXT),
       "--kv", KV,
       "--vision", VISION,
       "--vram-reserve-mib", str(VRAM_RESERVE_MIB),
       "--mtp", MTP,
       "--no-start", "--yes"]

if FAMILY in ("qwen", "coder") and DRAFT_VOCAB:
    cmd += ["--draft-vocab", DRAFT_VOCAB]
if ENGINE_SOURCE == "build":
    cmd += ["--build"]
if API_KEY:
    cmd += ["--api-key", API_KEY]

shown = " ".join("****" if API_KEY and a == API_KEY else shlex.quote(a) for a in cmd)
print("$", shown)
rc = subprocess.call(cmd)
print("setup exit code:", rc)


## 10. Tune the engine for this machine (optional)

Runs Strata's own tuner (`strata tune auto`) once, before the server starts, and saves the result as a preset. The server applies the active preset on every start, so no other cell has to change.

- `RUN_TUNE="auto"`: tune when there is no valid preset (first session, or the engine/model changed). A changed engine is rechecked in ~2 minutes first, and retuned only if the preset no longer helps.
- `RUN_TUNE="recheck"`: only recheck a cached preset; never a full tune.
- `RUN_TUNE="off"`: skip entirely.
- The preset travels in `strata-runtime-cache` (`tuning/presets/`), so later sessions start tuned at no cost. A stale preset is never applied blindly: Strata checks the engine, GPU, memory and model file first.
- A full tune is ~20-30 minutes on a desktop, longer on a shared T4; `TUNE_MINUTES` caps it. On failure the standard settings stay.


In [ ]:
import json
import subprocess
from pathlib import Path

STRATA = Path(STRATA_DIR)
PY = STRATA / ".venv" / "bin" / "python"
CLI = STRATA / "strata.py"
CFG = Path(CONFIG_PATH)

def _cli(args, timeout):
    """Run the Strata CLI; never raises."""
    try:
        p = subprocess.run([str(PY), str(CLI), *args], cwd=str(STRATA),
                           capture_output=True, text=True, timeout=timeout)
        return p.returncode, p.stdout or "", p.stderr or ""
    except Exception as exc:
        return 1, "", f"{type(exc).__name__}: {exc}"

def _json(text):
    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else {}
    except Exception:
        return {}

def _busy():
    return subprocess.run(["pgrep", "-f", "serve/server.py"],
                          stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0

if RUN_TUNE not in ("recheck", "auto"):
    print("tuning: RUN_TUNE =", RUN_TUNE, "-> skipped")
elif not (PY.is_file() and CLI.is_file() and CFG.is_file()):
    print("tuning: skipped (the setup cell has not finished yet)")
elif _busy():
    print("tuning: a server is already running -> skipped (stop it and re-run this cell to tune)")
else:
    rc, out, err = _cli(["presets", "status", str(CFG), "--output", "json"], 180)
    status = _json(out)
    state = status.get("state") if status else "unreadable"
    active = status.get("active") or {}
    if not status:
        print("tuning: could not read the preset store:",
              (err.strip()[-300:] or out.strip()[-300:] or "no output"))
    print("tuning: preset:", active.get("name") or "(none)",
          "| state:", state or "(none)",
          ("| " + "; ".join(status.get("reasons") or []) if status.get("reasons") else ""))

    full = False
    if RUN_TUNE == "auto" and state in (None, "hard"):
        full = True
    elif state == "soft" and active and active.get("id"):
        # the engine or driver changed: the designed cheap response is a recheck
        rc, out, err = _cli(["presets", "recheck", active["id"], str(CFG), "--output", "json"], 2400)
        data = _json(out)
        print("tuning: recheck ->", data.get("verdict") or (err.strip()[-300:] or "no answer"))
        full = (RUN_TUNE == "auto" and data.get("verdict") == "retune")
    elif state is None and RUN_TUNE == "recheck":
        print("tuning: no saved preset; RUN_TUNE='recheck' never runs a full tune (use 'auto' once)")
    elif state == "hard" and RUN_TUNE == "recheck":
        print("tuning: the engine/model changed too much for a recheck; set RUN_TUNE='auto' to tune again")

    if full:
        use = TUNE_USE if TUNE_USE in ("chat", "coding") else "chat"
        args = ["tune", "auto", str(CFG), "--mode", "quick", "--use", use, "--output", "json"]
        if TUNE_MINUTES and TUNE_MINUTES > 0:
            args += ["--time", str(TUNE_MINUTES)]
        budget = (int(TUNE_MINUTES) if TUNE_MINUTES else 30) * 60 + 900
        print("tuning: measuring for up to", TUNE_MINUTES or "the tuner's estimate", "min ...")
        rc, out, err = _cli(args, budget)
        result = _json(out).get("result") or {}
        if result:
            print("tuning:", result.get("outcome"),
                  "| gain", result.get("gain_pct"), "%",
                  "| preset", (result.get("preset") or {}).get("name"))
        if rc == 130:
            print("tuning: stopped; the standard settings stay")
        elif rc != 0:
            print("tuning: did not finish:", (err.strip()[-600:] or "no error output"))


## 11. Start the server (non-blocking)

This cell returns. The server keeps running in the background, so you can run the metrics cell afterwards.

In [ ]:
import os
import re
import subprocess
import time
import urllib.request
from pathlib import Path

for pattern in ("serve/server.py", "cloudflared", "ngrok"):
    subprocess.run(["pkill", "-f", pattern],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

if not Path(CONFIG_PATH).is_file():
    raise SystemExit(f"no config at {CONFIG_PATH} - run the setup cell first")

# Named tunnel: accept a bare host or a full URL in PUBLIC_HOSTNAME.
if PUBLIC_HOSTNAME:
    PUBLIC_HOSTNAME = PUBLIC_HOSTNAME.strip().rstrip("/")
    if not re.match(r"^https?://", PUBLIC_HOSTNAME):
        PUBLIC_HOSTNAME = "https://" + PUBLIC_HOSTNAME

def tree_rchar(pid):
    """Bytes read by the server and the engine process it starts.

    serve/server.py spawns the engine, and the engine is what reads the tens of
    GB of weights, so reading only the wrapper's /proc/<pid>/io shows 0 MB/s.
    """
    pids, children = {pid}, {}
    try:
        for line in subprocess.run(["ps", "-eo", "pid=,ppid="], capture_output=True,
                                   text=True, timeout=15).stdout.splitlines():
            child, parent = line.split()[:2]
            children.setdefault(int(parent), []).append(int(child))
    except Exception:
        pass
    stack = [pid]
    while stack:
        for child in children.get(stack.pop(), []):
            pids.add(child)
            stack.append(child)
    total = 0
    for current in pids:
        try:
            for line in Path(f"/proc/{current}/io").read_text().splitlines():
                if line.startswith("rchar:"):
                    total += int(line.split()[1])
        except Exception:
            pass
    return total


env = {**os.environ, "PYTHONUNBUFFERED": "1", "STRATA_REQUEST_LINES": "1"}
# Every name a client may use has to pass the server's DNS-rebinding check, and an
# API key does not switch that check off: a tunnel request still arrives with the
# public Host header. Local probes use 127.0.0.1, quick tunnels use
# *.trycloudflare.com, and a named tunnel adds PUBLIC_HOSTNAME.
allowed = ["127.0.0.1", "localhost", "127.0.0.1:" + str(PORT),
           "localhost:" + str(PORT), ".trycloudflare.com"]
if PUBLIC_HOSTNAME:
    named = PUBLIC_HOSTNAME.split("://", 1)[-1].split("/", 1)[0]
    allowed += [named, named + ":" + str(PORT), "." + named]
env["STRATA_ALLOWED_HOSTS"] = ",".join(allowed)
print("allowed hosts:", env["STRATA_ALLOWED_HOSTS"])

log = open(LOG_PATH, "w")
SERVER = subprocess.Popen(
    [f"{STRATA_DIR}/.venv/bin/python", "serve/server.py", "--engine", "strata",
     "--config", CONFIG_PATH, "--port", str(PORT)],
    cwd=STRATA_DIR, stdout=log, stderr=subprocess.STDOUT, env=env)
log.close()
print("server PID:", SERVER.pid, "->", LOG_PATH)

BASE_URL = f"http://127.0.0.1:{PORT}"
started = time.time()
last_read, last_time = 0, time.time()
healthy = False
while time.time() - started < HEALTH_TIMEOUT_S:
    if SERVER.poll() is not None:
        print("server exited early, code", SERVER.returncode)
        break
    try:
        body = urllib.request.urlopen(BASE_URL + "/health", timeout=10).read().decode()
        print("health OK:", body[:300])
        healthy = True
        break
    except Exception:
        time.sleep(5)
        elapsed = int(time.time() - started)
        if elapsed % 30 < 5:
            read = tree_rchar(SERVER.pid)
            now = time.time()
            rate = (read - last_read) / (now - last_time) / 1e6 if last_read else 0.0
            last_read, last_time = read, now
            vram = ""
            try:
                vram = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                                       "--format=csv,noheader,nounits"],
                                      capture_output=True, text=True, timeout=15).stdout.strip()
                vram = vram.replace("\n", " + ")
            except Exception:
                pass
            print(f"  loading... {elapsed}s  read {read / 1e9:.1f} GB at {rate:.0f} MB/s"
                  + (f"  vram {vram} MiB" if vram else ""))

if not healthy:
    print("health FAILED after", HEALTH_TIMEOUT_S, "s - last log lines:")
    print(Path(LOG_PATH).read_text(errors="ignore")[-3000:])
    print("The server is still running, so do not re-run this cell: it kills and restarts")
    print("the load. A start that reached 'filling the GPU expert cache' is working; it")
    print("streams the experts from the /kaggle/input mount, which is the slow part.")
    print("Poll it in a scratch cell instead:")
    print("  import urllib.request")
    print("  print(urllib.request.urlopen('" + BASE_URL + "/health', timeout=10).read().decode())")


## 12. Tunnel and client URL

Starts `cloudflared` in the background (quick or named tunnel), waits until it
registers with the Cloudflare edge, prints the public URL, and probes it end to
end. Does not block the kernel.

With `CLOUDFLARE_TUNNEL_TOKEN` set the tunnel is a **named** one:

- `run --token` ignores a local `config.yml`. Add the hostname under
  **Zero Trust → Networks → Tunnels → your tunnel → Public Hostname**, service
  `HTTP` at `127.0.0.1:<PORT>` (default `8080`).
- Set `PUBLIC_HOSTNAME` to that name. The scheme is optional:
  `strata.cyberctl.com` is normalized to `https://strata.cyberctl.com`.
- Keep `API_KEY` set. A fixed hostname is public and guessable, and the key is
  what stops other people from using the model. Without it, the server cell adds
  the hostname to `STRATA_ALLOWED_HOSTS` so the DNS-rebinding check passes.
- Cloudflare errors 1033/502/1016 come from the tunnel or the zone, not from
  Strata; a 403 is either a Zero Trust Access policy or Strata's host check.
  The probe prints who answered, so the two can be told apart.


In [ ]:
import re
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

PUBLIC_URL = None

if not globals().get("healthy"):
    print("skipped: the server is not healthy")
elif not ENABLE_TUNNEL:
    print("tunnel disabled; local URL:", BASE_URL)
else:
    Path(CF_LOG_PATH).write_text("")
    if TUNNEL_TOKEN:
        cf_cmd = ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", TUNNEL_TOKEN]
        PUBLIC_URL = PUBLIC_HOSTNAME or None
    else:
        cf_cmd = ["cloudflared", "tunnel", "--no-autoupdate", "--url", BASE_URL]

    cf_log = open(CF_LOG_PATH, "w")
    CF_PROC = subprocess.Popen(cf_cmd, stdout=cf_log, stderr=subprocess.STDOUT)
    cf_log.close()

    def cf_text():
        return Path(CF_LOG_PATH).read_text(errors="ignore")

    # A named tunnel only serves traffic once it has registered with the edge.
    connected = False
    started = time.time()
    while time.time() - started < 120:
        if "Registered tunnel connection" in cf_text():
            connected = True
            break
        if CF_PROC.poll() is not None:
            break
        time.sleep(3)

    if not connected and TUNNEL_TOKEN:
        print("cloudflared did NOT register with the Cloudflare edge. Log tail:")
        print(cf_text()[-3000:])
        print()
        print("Checks: the token belongs to this tunnel and was not rotated; the tunnel has")
        print("a Public Hostname for " + (PUBLIC_HOSTNAME or "(PUBLIC_HOSTNAME is empty)")
              + " with service HTTP at http://127.0.0.1:" + str(PORT) + ".")
        print("A token run ignores any local config.yml: the hostname and its service URL are")
        print("configured in the Cloudflare dashboard, not in the notebook.")
    elif connected:
        print("cloudflared: registered with the edge")
    elif CF_PROC.poll() is not None:
        print("cloudflared exited early - log tail:")
        print(cf_text()[-3000:])

    # Quick tunnels publish their generated hostname in the log.
    if not PUBLIC_URL and not TUNNEL_TOKEN:
        started = time.time()
        while time.time() - started < 90:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_text())
            if m:
                PUBLIC_URL = m.group(0)
                break
            time.sleep(3)

    if not PUBLIC_URL:
        if TUNNEL_TOKEN:
            print("named tunnel: set PUBLIC_HOSTNAME in the config cell, e.g.")
            print('  PUBLIC_HOSTNAME = "https://strata.cyberctl.com"')
        else:
            print("no public URL found - see", CF_LOG_PATH)
    else:
        print("public URL:", PUBLIC_URL)
        print("OpenAI base URL:", PUBLIC_URL + "/v1")
        if API_KEY:
            print("clients must send: Authorization: Bearer <API_KEY>")
        else:
            print("!! no API key: anyone with this URL can use the model")

        # End to end through the tunnel. A 403 can come from Cloudflare (an Access
        # policy) or from Strata (the host check), so report who answered.
        def probe(path, headers=None):
            request = urllib.request.Request(PUBLIC_URL + path, headers=headers or {})
            try:
                with urllib.request.urlopen(request, timeout=30) as response:
                    return response.status, response.headers, ""
            except urllib.error.HTTPError as exc:
                return exc.code, exc.headers, exc.read(2000).decode(errors="replace")
            except Exception as exc:
                return None, None, type(exc).__name__ + ": " + str(exc)

        status, headers, body = probe("/health")
        if status == 200:
            print("tunnel check: /health HTTP 200 -> reachable")
        else:
            print("tunnel check: /health", status if status is not None else "failed")
            if headers is not None:
                print("  content-type:", headers.get("content-type"),
                      "| server:", headers.get("server"),
                      "| cf-mitigated:", headers.get("cf-mitigated"))
            if body:
                print("  body:", " ".join(body.split())[:200])
            html = headers is not None and "text/html" in (headers.get("content-type") or "")
            challenged = headers is not None and bool(headers.get("cf-mitigated"))
            if html or challenged:
                print("  Cloudflare answered, not Strata: a Zero Trust Access policy guards")
                print("  this hostname. Add a bypass or a service token for /v1, or remove it.")
            elif status in (400, 403):
                print("  Strata answered and refused: the Host header is not in")
                print("  STRATA_ALLOWED_HOSTS, or /health needs the API key in this build.")
            elif status is not None and status >= 500:
                print("  Cloudflare reached the edge but not the server: fix the tunnel's")
                print("  Public Hostname service URL, it must be http://127.0.0.1:" + str(PORT))
            elif status is None:
                print("  DNS/zone: the hostname's domain must be an active Cloudflare zone")
                print("  in this account, and the rule must exist on this tunnel.")

        if API_KEY:
            status, headers, body = probe("/v1/models", {"Authorization": "Bearer " + API_KEY})
            print("tunnel check: /v1/models", status if status is not None else "failed",
                  "(authenticated)")
            if status == 200:
                print("  the API key is accepted end to end")
            elif status in (401, 403):
                print("  the key was rejected: compare API_KEY with the key setup received")
                print("  (cell 9 passes it as --api-key) and check --access cloudflare")
            elif body:
                print("  body:", " ".join(body.split())[:200])


## 13. Per-request numbers (run after a few messages)

`prompt_ms` is the prefill wait, `decode_tok_s` the generation speed, `file_mb` how much was read from disk, `hit_rate` the expert-cache hit ratio. Compare runs to see whether the VRAM levers helped.

In [ ]:
import json
import urllib.request

def show_metrics(n=6):
    request = urllib.request.Request(BASE_URL + "/metrics")
    if API_KEY:
        request.add_header("Authorization", "Bearer " + API_KEY)
    data = json.loads(urllib.request.urlopen(request, timeout=30).read().decode())
    rows = data.get("requests", [])[:n]
    if not rows:
        print("no finished requests yet")
        return
    print(f"{'out':>6} {'tok/s':>7} {'prompt_ms':>10} {'hit':>6} {'file_MB':>8} {'drafts':>12}")
    for r in rows:
        drafts = f"{r.get('drafts_accepted', 0)}/{r.get('drafts_offered', 0)}"
        print(f"{r.get('output_tokens', 0):6} {str(r.get('decode_tok_s')):>7} "
              f"{str(r.get('prompt_ms')):>10} {str(r.get('hit_rate')):>6} "
              f"{str(r.get('file_mb')):>8} {drafts:>12}")

try:
    show_metrics()
except Exception as exc:
    print("metrics unavailable:", exc)


## 14. Diagnostics — disk, RAM, GPUs

Prints disk, pack/MTP/model sizes, RAM and GPU utilisation. To stop the model: `pkill -f serve/server.py ; pkill -f cloudflared`.

In [ ]:
import subprocess

for cmd in (
    ["df", "-h", "/root", "/kaggle/input"],
    ["du", "-sh", f"{DATA_DIR}/packs", f"{DATA_DIR}/mtp", f"{DATA_DIR}/models"],
    ["free", "-h"],
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader"],
):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)
    print()

print("$ tail -20", CF_LOG_PATH)
subprocess.run(["tail", "-20", CF_LOG_PATH], check=False)
print()

print("to stop the model:  pkill -f serve/server.py ; pkill -f cloudflared")


## 15. Build the caches (one-time, optional)

Set `UPLOAD_CACHE = True` in the config cell and run this cell after setup. It publishes:

- `strata-runtime-cache` — `engine/` + `third_party/llama.cpp` + `cloudflared`, as `strata-runtime-cache.bin` (an uncompressed tar, so Kaggle does not unpack it) with a `.sha256` and a `runtime-manifest.json` (engine version, archs, repo commit, GPUs)
- `strata-<tag>-pack` — the pack, `experts.bin` included (~52 GB for IQ3_S, ~25 GB for Coder)
- `strata-mtp-rt` — `mtp/rt` + `mtp-q2_0.gguf` (~1.7 GB, shared by every variant)

The pack and MTP are staged as symlinks, so no extra disk is used; the engine is a real tar (~50–150 MB). Wait for **Ready** on each Kaggle dataset page before ending the session, then set `UPLOAD_CACHE` back to `False`.

In [ ]:
if not UPLOAD_CACHE:
    print("UPLOAD_CACHE is False - nothing uploaded")
else:
    import hashlib
    import io
    import json as _json
    import os
    import platform
    import re as _re
    import shutil
    import subprocess
    import sys
    import tarfile as _tarfile
    import time
    from pathlib import Path

    if not KAGGLE_TOKEN:
        raise SystemExit("the KAGGLE_API_TOKEN secret is required for upload")

    home = Path("/tmp/kaggle-cli-home")
    cfg = home / ".kaggle"
    cfg.mkdir(parents=True, exist_ok=True)
    token = (KAGGLE_TOKEN or "").strip()
    cli_env = {**os.environ, "HOME": str(home), "KAGGLE_CONFIG_DIR": str(cfg),
               "KAGGLE_USERNAME": KAGGLE_USERNAME, "TMPDIR": str(Path("/kaggle/working/tmp"))}
    if _re.fullmatch(r"[0-9a-fA-F]{32}", token):
        # legacy API key: the CLI wants the username beside it
        (cfg / "kaggle.json").write_text(
            _json.dumps({"username": KAGGLE_USERNAME, "key": token}))
        (cfg / "kaggle.json").chmod(0o600)
    else:
        # an access token (KGAT_...): the file and the variable the CLI reads
        (cfg / "access_token").write_text(token + "\n")
        (cfg / "access_token").chmod(0o600)
        cli_env["KAGGLE_API_TOKEN"] = token
    tmp = Path(cli_env["TMPDIR"])
    tmp.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
                   env=cli_env, check=False)

    _cli = shutil.which("kaggle")

    def kaggle(*args):
        cmd = [_cli] if _cli else [sys.executable, "-m", "kaggle"]
        return subprocess.call([*cmd, *args], env=cli_env)

    def upload(stage, dataset, title, description):
        (stage / "dataset-metadata.json").write_text(_json.dumps(
            {"title": title, "id": dataset, "licenses": [{"name": "other"}],
             "description": description}, indent=2))
        rc = kaggle("datasets", "create", "-p", str(stage), "--dir-mode", "skip")
        if rc != 0:
            rc = kaggle("datasets", "version", "-p", str(stage),
                        "-m", "update", "--dir-mode", "skip")
        if rc != 0:
            print(f"  !! {dataset}: the upload failed (see the messages above)")
        else:
            print(f"  {dataset}: uploaded - wait for 'Ready' on its Kaggle page "
                  f"(https://www.kaggle.com/datasets/{dataset})")

    def publish_pack(dataset, title, files):
        if not dataset:
            print("no dataset configured (None) - skipped")
            return
        if not files:
            print("nothing to upload for", dataset)
            return
        stage = Path("/kaggle/working") / ("stage-" + dataset.rpartition("/")[2])
        if stage.exists():
            shutil.rmtree(stage)
        stage.mkdir(parents=True)

        # Every entry is a *file* symlink at the top level (flattened, NNNN__path__name):
        # `--dir-mode skip` ignores directories, so a nested layout would upload nothing,
        # and Kaggle allows at most 50 top-level entries (a pack has ~11).
        entries = []
        for i, path in enumerate(sorted(files), 1):
            rel = path.relative_to(Path(DATA_DIR)).as_posix()
            name = f"{i:04d}__" + _re.sub(r"[^A-Za-z0-9._-]+", "__", rel.replace("/", "__"))
            (stage / name).symlink_to(path.resolve())
            entries.append({"dataset_name": name, "relative_path": rel,
                            "size": path.stat().st_size})

        (stage / "manifest.json").write_text(_json.dumps(
            {"format": "strata-cache.v1", "model": {"family": FAMILY, "model": MODEL},
             "files": entries}, indent=2))
        total = sum(e["size"] for e in entries) / 2 ** 30
        if total > 200:
            print(f"!! {dataset}: {total:.1f} GiB exceeds Kaggle's 200 GB per-dataset limit")
        if len(entries) + 2 > 50:
            print(f"!! {dataset}: {len(entries)} files + manifest + metadata exceeds Kaggle's 50 top-level limit")
        print(f"{dataset}: {len(entries)} files, {total:.1f} GiB")
        upload(stage, dataset, title, "Private Strata cache. Upstream model terms apply.")

    def publish_runtime():
        if not RUNTIME_DATASET:
            print("RUNTIME_DATASET is None - runtime upload skipped")
            return
        engine_dir = Path(STRATA_DIR) / "engine"
        if not (engine_dir / "strata").is_file():
            print("runtime: no engine installed - skipped")
            return
        try:
            meta = _json.loads((engine_dir / "BUILD.json").read_text(encoding="utf-8"))
        except Exception:
            meta = {}

        # The restore cell validates a cached tree by engine/BUILD.json. When the build
        # wrote none, describe the binary here so this upload can still be reused.
        synthesized = None
        if not (engine_dir / "BUILD.json").is_file():
            try:
                version = subprocess.run([str(engine_dir / "strata"), "--version"],
                                         capture_output=True, text=True, timeout=60).stdout
            except Exception:
                version = ""
            found = _re.search(r"\d+\.\d+(?:\.\d+)?", version or "")
            caps = []
            try:
                for cap in subprocess.run(["nvidia-smi", "--query-gpu=compute_cap",
                                           "--format=csv,noheader"],
                                          capture_output=True, text=True, timeout=60).stdout.split():
                    m = _re.fullmatch(r"(\d+)\.(\d+)", cap.strip())
                    if m:
                        caps.append(int(m.group(1)) * 10 + int(m.group(2)))
            except Exception:
                caps = []
            synthesized = {"version": found.group(0) if found else "",
                           "source": "downloaded", "archs": sorted(set(caps))}
            meta = synthesized
            print("  engine/BUILD.json is missing; recording", synthesized, "in the archive")

        stage = Path("/kaggle/working/strata-runtime-dataset-stage")
        if stage.exists():
            shutil.rmtree(stage)
        stage.mkdir(parents=True)

        # ".bin" = an uncompressed tar.  Kaggle unpacks archive suffixes server-side;
        # a plain name keeps the tree exactly as we wrote it.
        archive = stage / "strata-runtime-cache.bin"

        def skip_big(ti):
            name = ti.name.rsplit("/", 1)[-1]
            if name.endswith(".zip") or name == "BUILD.json.prev":
                return None
            return ti

        presets_dir = Path(os.environ.get("XDG_CONFIG_HOME", str(Path.home() / ".config"))) / "strata" / "presets"
        preset_ids = sorted(p.stem for p in presets_dir.glob("*.json") if p.name != "active.json") \
            if presets_dir.is_dir() else []

        with _tarfile.open(archive, "w") as tf:
            tf.add(engine_dir, arcname="engine", filter=skip_big)
            if synthesized is not None:
                payload = _json.dumps(synthesized, indent=2).encode()
                info = _tarfile.TarInfo("engine/BUILD.json")
                info.size, info.mode = len(payload), 0o644
                tf.addfile(info, io.BytesIO(payload))
            llama = Path(STRATA_DIR) / "third_party" / "llama.cpp"
            if llama.is_dir():
                tf.add(llama, arcname="third_party/llama.cpp", filter=skip_big)
            cf = Path("/usr/local/bin/cloudflared")
            if cf.is_file():
                tf.add(cf, arcname="tools/cloudflared")
            if preset_ids:
                tf.add(presets_dir, arcname="tuning/presets", filter=skip_big)

        digest = hashlib.sha256()
        with open(archive, "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                digest.update(block)
        (stage / (archive.name + ".sha256")).write_text(
            digest.hexdigest() + "  " + archive.name + "\n", encoding="utf-8")

        try:
            commit = subprocess.run(["git", "-C", STRATA_DIR, "rev-parse", "HEAD"],
                                    capture_output=True, text=True, check=False).stdout.strip()
        except Exception:
            commit = ""
        try:
            gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                  capture_output=True, text=True, check=False).stdout.split("\n")
            gpus = [g.strip() for g in gpus if g.strip()]
        except Exception:
            gpus = []

        def arch_list(value):
            if value in (None, ""):
                return []
            parts = (list(value) if isinstance(value, (list, tuple, set))
                     else _re.split(r"[;,\s]+", str(value)))
            out = []
            for part in parts:
                m = _re.search(r"\d+", str(part))
                if m and int(m.group()) not in out:
                    out.append(int(m.group()))
            return out

        archs = []
        for key in ("archs", "engine_archs", "cuda_archs", "cmake_cuda_architectures",
                    "CUDA_ARCHITECTURES", "TORCH_CUDA_ARCH_LIST", "torch_cuda_arch_list"):
            archs = arch_list(meta.get(key))
            if archs:
                break

        manifest = {
            "format": "strata-kaggle.runtime.v1",
            "archive": archive.name,
            "engine_version": meta.get("version"),
            "engine_archs": archs,
            "engine_source": meta.get("source"),
            "backend": meta.get("backend"),
            "repo_commit": commit,
            "gpus": gpus,
            "python": platform.python_version(),
            "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "tuning_presets": preset_ids,
            "contents": ["engine", "third_party/llama.cpp (if present)", "tools/cloudflared (if present)",
                         "tuning/presets (if a preset was saved)"],
        }
        (stage / "runtime-manifest.json").write_text(_json.dumps(manifest, indent=2) + "\n",
                                                    encoding="utf-8")

        print(f"{RUNTIME_DATASET}: engine {meta.get('version')}, archs {archs}, "
              f"{archive.stat().st_size / 2 ** 20:.0f} MiB"
              + (f", {len(preset_ids)} tuning preset(s)" if preset_ids else ""))
        if not archs:
            print("  note: BUILD.json lists no GPU archs; the manifest records none")
        elif 75 not in archs:
            print("  warning: no sm_75 code - auto restore will skip this engine on a T4")
        if meta.get("source") == "local":
            print("  warning: local build - auto restore skips local builds; only cloudflared is reused")
            print("  to publish a restorable engine, run ENGINE_SOURCE=prebuilt or let setup download it")
        upload(stage, RUNTIME_DATASET, "Strata runtime cache (engine + cloudflared)",
               "Private Strata runtime. Upstream terms apply; includes the engine binary.")

    pack_files = [p for p in Path(PACK_DIR).rglob("*")
                  if p.is_file() and not p.name.endswith(".part")]
    publish_pack(PACK_DATASET, f"Strata {MODEL} pack", pack_files)

    mtp_root = Path(DATA_DIR) / "mtp"
    mtp_files = [p for p in (mtp_root / "rt").rglob("*")
                 if p.is_file() and not p.name.startswith("draft_vocab")]
    if (mtp_root / "mtp-q2_0.gguf").is_file():
        mtp_files.append(mtp_root / "mtp-q2_0.gguf")
    publish_pack(MTP_DATASET, "Strata MTP draft layer", mtp_files)

    publish_runtime()
